## Structured Output

Model can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. Langchain support multiple schema types and methods for enforcing structured output.

## Pydentic

Pydentic models provide the richest feature ste field validataion, descriptions and nested structures.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3.6-27b")

In [2]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director movie")
    rating:float=Field(description="The movies rating out of 10")

In [3]:
model_with_strcuture=model.with_structured_output(Movie)
model_with_strcuture

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.4', 'langchain': '1.3.15'}}, client=<groq.resources.chat.completions.Completions object at 0x10e519e80>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x10e51aba0>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director movie', 'type': 'string'}, 'rating': {'description': 'The movies rating out of 10', 'type': 'number'}}, 'required': ['title', 'year', 'director', 'rating'], 'type': 'object'}}}], 'ls_structured_output_format': {'kwargs': {'method': 'function_calling'}, 'schema': {'type': 'function', 'function': {'name': 'Movie', 'description': 

In [4]:
response = model.invoke("Provide details about the movie Inception")
response

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Understand User Request**: The user asks for "details about the movie Inception". This is a straightforward request for comprehensive information about the film.\n\n2.  **Identify Key Information Needed**:\n   - Title: Inception\n   - Release Year: 2010\n   - Director: Christopher Nolan\n   - Writers: Christopher Nolan\n   - Starring: Leonardo DiCaprio, Joseph Gordon-Levitt, Elliot Page, Tom Hardy, Ken Watanabe, Cillian Murphy, Marion Cotillard, Michael Caine\n   - Genre: Sci-fi, Action, Thriller\n   - Plot Summary: Core concept, main characters, mission, twists\n   - Themes: Dreams vs. reality, guilt, subconscious, time perception\n   - Production/Reception: Budget, box office, critical reception, awards\n   - Notable Elements: Visual effects, score (Hans Zimmer), spinning top ending, layered dream structure\n   - Legacy/Impact: Cultural impact, influence on cinema, sequels/spin-offs (none official, but discussions)\n\

In [5]:
response = model_with_strcuture.invoke("Provide details about the movie Inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

In [6]:
response = model_with_strcuture.invoke("Provide details about the movie Avatar")
response

Movie(title='Avatar', year=2009, director='James Cameron', rating=7.9)

In [7]:
response = model_with_strcuture.invoke("Provide details about the movie Interstellar")
response

Movie(title='Interstellar', year=2014, director='Christopher Nolan', rating=8.7)

### Message Output alongside Parsed structure

In [8]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The sirector of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)

response = model_with_structure.invoke("Provide details about movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:** The user is asking for details about the movie "Inception".\n2.  **Identify Required Information:** To use the `Movie` function, I need:\n   - title: "Inception"\n   - year: 2010 (I know this from general knowledge)\n   - director: Christopher Nolan (I know this)\n   - rating: I need a rating out of 10. Let\'s check common ratings. IMDb rating is around 8.8/10. I\'ll use 8.8.\n3.  **Verify Function Parameters:** The `Movie` function requires:\n   - title (string)\n   - year (integer)\n   - director (string)\n   - rating (number)\n   All are available/known.\n4.  **Construct Function Call:**\n   ```json\n   {\n     "name": "Movie",\n     "parameters": {\n       "title": "Inception",\n       "year": 2010,\n       "director": "Christopher Nolan",\n       "rating": 8.8\n     }\n   }\n   ```\n5.  **Execute Function Call:** I will call the function with these para

### Nested Structure

In [9]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    gonres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

In [10]:
model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Marion Cotillard', role='Mal')], gonres=['Action', 'Adventure', 'Sci-Fi', 'Thriller'], budget=None)

## TypedDict

TypedDict provides a simpler alternative using Python's built-in typing, ideal when you don't need runtime validaion.

In [12]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with detaiils."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


In [15]:
model_with_typedict = model.with_structured_output(MovieDict)

model_with_typedict.invoke("Please provide the details of the movie avengers")

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}

In [17]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    gonres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Avengers")
response

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'}],
 'gonres': ['Action', 'Adventure', 'Sci-Fi'],
 'title': 'Avengers',
 'year': 2012}

In [18]:
model_with_typedict.profile

AttributeError: 'RunnableSequence' object has no attribute 'profile'

In [22]:
model.profile

## DataClasses

A data class is a class typically containing mainly data, although there aren't really any restrictions. You create it using @dataclass decorator.

In [23]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3.6-27b")

In [28]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a perosn."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phobe number of the person")

agent = create_agent(
    model = "groq:qwen/qwen3.6-27b",
    response_format = ContactInfo # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

print(result["structured_response"])
# contactinfo(name='john Doe', email='john@example.com', phone='(555) 123-4567')

name='John Doe' email='john@example.com' phone='(555) 123-4567'


In [29]:
result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='2cf0a73c-0ed8-42cd-ab2b-11ffa547c68a'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User wants to extract contact info from the string: "John Doe, john@example.com, (555) 123-4567"\n   - The available function is `ContactInfo` which requires `name`, `email`, and `phone`.\n\n2.  **Map Input to Function Parameters:**\n   - Name: "John Doe"\n   - Email: "john@example.com"\n   - Phone: "(555) 123-4567"\n\n3.  **Check Function Requirements:**\n   - Function: `ContactInfo`\n   - Required parameters: `name`, `email`, `phone`\n   - All required parameters are present in the input.\n\n4.  **Construct Function Call:**\n   ```json\n   {\n     "name": "ContactInfo",\n     "arguments": {\n       "name": "John Doe",\n       "email": "john@example.com",\n       "

In [30]:
## TypeDIct
from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """Contact information for a perosn."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phobe number of the person

agent = create_agent(
    model = "groq:qwen/qwen3.6-27b",
    response_format = ContactInfo # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

print(result["structured_response"])
# contactinfo(name='john Doe', email='john@example.com', phone='(555) 123-4567')

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}


In [31]:
## DataClass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model = "groq:qwen/qwen3.6-27b",
    response_format = ContactInfo # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

print(result["structured_response"])

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')
